In [11]:
!uv pip install -q pandas plotly
!uv pip install -q kaleido nbformat

In [12]:
import pandas as pd
import glob
import json
import plotly.express as px

def load_data(paths):
    files = []
    for p in paths:
        files.extend(glob.glob(f"{p}/**/result.json", recursive=True))
    return pd.DataFrame([json.load(open(f)) for f in files])

def clean_model_ids(model_ids: pd.Series):
    return model_ids.str.split('/').str[-1].str.replace("Instruct", "it").str.replace("b", "B").str.replace("270m", "270M").str.replace("-it", "")

In [13]:
paths = [
    './multirun/head-final',
]
df = load_data(paths)
df = df.round(6)
df = df.drop(columns=['times', 'autorange_min_run_time'])
df = df.sort_values(by=['model_id'], ascending=True, key=lambda col: col.str.lower())
model_order = ['gemma-3-270M', 'gemma-3-1B', 'Llama-3.2-1B', 'Llama-3.2-3B', 'Qwen3-0.6B', 'Qwen3-1.7B']

df["model_id"] = clean_model_ids(df["model_id"])
df = df[~(df['model_id'] == 'Llama-3.2-3B')]
df.head()

,model_id,ef,batch_size,is_ref,k,benchmark_lm_head_only,sleep,mean,median,p25,p75,iqr,num_threads
14,gemma-3-1B,200,1,False,50,True,4.0,0.002728,0.002736,0.002636,0.002806,0.000169,6
15,gemma-3-1B,200,1,True,50,True,4.0,0.068432,0.068674,0.068544,0.068855,0.000311,6
16,gemma-3-1B,200,2,False,50,True,4.0,0.004019,0.003991,0.003705,0.004276,0.000571,6
17,gemma-3-1B,200,2,True,50,True,4.0,0.055733,0.054541,0.054132,0.054833,0.000701,6
18,gemma-3-1B,200,4,False,50,True,4.0,0.006046,0.005989,0.005729,0.006273,0.000544,6


In [14]:
def compare_runs(df, index_cols):
    df = df.groupby([*index_cols, 'is_ref']).first().reset_index()

    ref_median = df[df['is_ref'] == True].set_index(index_cols)[['median']]
    opt_median = df[df['is_ref'] == False].set_index(index_cols)[['median']]

    speedup = ref_median / opt_median

    dfs = ref_median.join(opt_median, lsuffix='_ref', rsuffix='_nonref')
    dfs['speedup'] = speedup

    return dfs.reset_index()

dfs = compare_runs(df, index_cols=['model_id', 'ef', 'batch_size'])
dfs[dfs["batch_size"] == 1]

,model_id,ef,batch_size,median_ref,median_nonref,speedup
0,Llama-3.2-1B,200,1,0.059821,0.003946,15.159909
7,Qwen3-0.6B,200,1,0.035532,0.002721,13.058434
14,Qwen3-1.7B,200,1,0.071026,0.004472,15.882379
21,gemma-3-1B,200,1,0.068674,0.002736,25.100146
28,gemma-3-270M,200,1,0.038316,0.001623,23.608133


In [15]:
paper_style_dict = {
    'layout.plot_bgcolor': 'rgba(0, 0, 0, 0)',
    'layout.font.family': 'monospace', # 'Times New Roman',
    'layout.font.size': 20,
    'layout.xaxis.linecolor': 'black',
    'layout.xaxis.ticks': 'outside',
    'layout.xaxis.mirror': True,
    'layout.xaxis.showline': True,
    'layout.xaxis.showgrid': True,
    'layout.xaxis.gridcolor': 'lightgray',
    'layout.xaxis.gridwidth': 0.1,
    'layout.yaxis.linecolor': 'black',
    'layout.yaxis.ticks': 'outside',
    'layout.yaxis.mirror': True,
    'layout.yaxis.showline': True,
    'layout.yaxis.showgrid': True,
    'layout.yaxis.gridcolor': 'lightgray',
    'layout.yaxis.gridwidth': 0.1,
    'layout.autosize': False,
    'layout.showlegend': True,
    'layout.legend.bgcolor': 'rgba(0, 0, 0, 0)',
    'layout.legend.xanchor': 'right',
    'layout.legend.x': 0.98,
    'layout.legend.yanchor': 'top',
    'layout.legend.y': 0.98,
    'layout.legend.font.family': 'monospace',
    'layout.legend.font.size': 16,
    'layout.xaxis.gridwidth': 1, 
    'layout.yaxis.gridwidth': 1,
    'layout.margin': {'l': 20, 'r': 20, 't': 20, 'b': 20},
}

In [16]:
def plot(df, ylim_max=None):
    fig = px.line(
        df, 
        x="batch_size", 
        y="speedup",
        range_y=[0, ylim_max],
        range_x=[1, 32],
        color="model_id",
        markers=True,
        color_discrete_sequence=px.colors.qualitative.Dark2,
        category_orders={'model_id': model_order},
        labels={
            "batch_size": "Batch Size",
            "speedup": "Speedup (×)",
            "model_id": "Model Architecture"
        }
    )

    fig.add_hline(
        y=1, 
        line_dash="dash", 
        line_color="black", 
        annotation_text="Baseline", 
        annotation_position="top left",
        line_width=1
    )


    fig.update_layout(xaxis = dict(tickvals = list(df["batch_size"])))

    fig.update_yaxes(dtick=2)

    fig.update(**paper_style_dict)
    return fig

dfs.to_csv("result/head-speedup.csv")

#print("EF 100")
#fig = plot(dfs[(dfs["ef"] == 100)], ylim_max=35)
#fig.write_image("head-speedup-ef-100.pdf")
#fig.show()

print("EF 200")
fig = plot(dfs[(dfs["ef"] == 200)], ylim_max=25)
fig.write_image("head-speedup-ef-200.pdf")

# fig.update({**paper_style_dict, 'layout.width': 1200, 'layout.font.size': 24, 'layout.legend.font.size': 18})
# fig.write_image("head-speedup-ef-200.svg")
fig.show()

EF 200


## Full Model

In [17]:
paths = [
    './multirun/full-final-part-qwen',
    './multirun/full-final',
]
df = load_data(paths)
df = df.round(6)
print(df)
df = df.drop(columns=['times', 'autorange_min_run_time'])
df = df.sort_values(by=['model_id'], ascending=True, key=lambda col: col.str.lower())
df["model_id"] = clean_model_ids(df["model_id"])

df = df[~(df['model_id'] == 'Llama-3.2-3B')]

dfs = compare_runs(df, index_cols=['model_id', 'ef', 'batch_size', 'new_token_count'])
dfs.head()

                            model_id  batch_size  is_ref   ef  \
0                    Qwen/Qwen3-0.6B           1   False  200   
1                    Qwen/Qwen3-0.6B           1    True  200   
2                    Qwen/Qwen3-0.6B           2   False  200   
3                    Qwen/Qwen3-0.6B           2    True  200   
4                    Qwen/Qwen3-0.6B           4   False  200   
..                               ...         ...     ...  ...   
79  meta-llama/Llama-3.2-3B-Instruct           8    True  200   
80  meta-llama/Llama-3.2-3B-Instruct          16   False  200   
81  meta-llama/Llama-3.2-3B-Instruct          16    True  200   
82  meta-llama/Llama-3.2-3B-Instruct          32   False  200   
83  meta-llama/Llama-3.2-3B-Instruct          32    True  200   

    new_token_count   k  autorange_min_run_time  benchmark_lm_head_only  \
0               128  50                      60                   False   
1               128  50                      60                   Fal

,model_id,ef,batch_size,new_token_count,median_ref,median_nonref,speedup
0,Llama-3.2-1B,200,1,128,22.389709,18.986438,1.179247
1,Llama-3.2-1B,200,2,128,34.368985,29.759840,1.154878
2,Llama-3.2-1B,200,4,128,36.205471,31.724577,1.141244
3,Llama-3.2-1B,200,8,128,41.295520,37.638541,1.097160
4,Llama-3.2-1B,200,16,128,51.154664,48.613834,1.052266


In [18]:
df[df["model_id"] == 'gemma-3-270M']

,model_id,batch_size,is_ref,ef,new_token_count,k,benchmark_lm_head_only,compile,warmup,sleep,mean,median,p25,p75,iqr,num_threads
12,gemma-3-270M,1,False,200,128,50,False,True,2,8,2.744105,2.754315,2.734567,2.771049,0.036482,6
13,gemma-3-270M,1,True,200,128,50,False,True,2,8,5.026088,5.024611,5.022075,5.026021,0.003946,6
14,gemma-3-270M,2,False,200,128,50,False,True,2,8,3.547726,3.541274,3.532740,3.557163,0.024424,6
15,gemma-3-270M,2,True,200,128,50,False,True,2,8,8.074271,8.072288,8.067838,8.079387,0.011549,6
16,gemma-3-270M,4,False,200,128,50,False,True,2,8,5.668674,5.656337,5.619298,5.727614,0.108316,6
17,gemma-3-270M,4,True,200,128,50,False,True,2,8,8.454801,8.453016,8.449648,8.457822,0.008174,6
18,gemma-3-270M,8,False,200,128,50,False,True,2,8,6.621392,6.622635,6.611600,6.635400,0.023800,6
19,gemma-3-270M,8,True,200,128,50,False,True,2,8,9.475310,9.474881,9.470581,9.480561,0.009981,6
20,gemma-3-270M,16,False,200,128,50,False,True,2,8,7.842931,7.834141,7.813052,7.880779,0.067727,6
21,gemma-3-270M,16,True,200,128,50,False,True,2,8,10.744919,10.741937,10.739831,10.745348,0.005516,6


In [19]:
dfs = dfs[dfs["batch_size"] != 2]

def plot(df):
    fig = px.line(
        df, 
        x="batch_size", 
        y="speedup",
        range_x=[1, 32],
        range_y=[0.95, 1.85],
        color="model_id",
        markers=True,
        color_discrete_sequence=px.colors.qualitative.Dark2,
        category_orders={'model_id': model_order},
        labels={
            "batch_size": "Batch Size",
            "speedup": "Speedup (×)",
            "model_id": "Model Architecture"
        }
    )

    fig.add_hline(
        y=1, 
        line_dash="dash", 
        line_color="black", 
        annotation_text="Baseline", 
        annotation_position="top left",
        line_width=1
    )


    fig.update_layout(xaxis = dict(tickvals = list(df["batch_size"])))
    fig.update_yaxes(dtick=0.2)

    fig.update(**paper_style_dict)
    return fig

dfs.to_csv("result/full-speedup.csv")

print("EF 200")
fig = plot(dfs)
fig.write_image("full-speedup-ef-200.pdf")

# fig.update({**paper_style_dict, 'layout.width': 1200, 'layout.font.size': 24, 'layout.legend.font.size': 18})
# fig.write_image("full-speedup-ef-200.svg")
fig.show()

EF 200


## Recall

In [ ]:
paths = ['./multirun/recall-final']
df = load_data(paths)
df = df.round(6)
df = df.drop(columns=["print_error"])
model_order = ['gemma-3-270M', 'gemma-3-1B', 'Qwen3-0.6B', 'Qwen3-1.7B', 'Llama-3.2-1B', 'Llama-3.2-3B']

df["model_id"] = clean_model_ids(df["model_id"])
df['model_id'] = pd.Categorical(df['model_id'], categories=model_order, ordered=True)
df = df.sort_values('model_id')

recall_cols = df.filter(like='average_top').columns
df[recall_cols] = (df[recall_cols] * 100).round(1)

recall_cols = df.filter(like='recall').columns
df[recall_cols] = (df[recall_cols] * 100).round(1)

df.to_csv("result/recall-result.csv")
df

In [ ]:
df[["model_id", "average_top1_token_probability", "recall-top-1-ef-100", "recall-top-1-ef-200", "recall-top-1-ef-400"]]

,model_id,average_top1_token_probability,recall-top-1-ef-100,recall-top-1-ef-200,recall-top-1-ef-400
2,gemma-3-270M,64.8,97.7,99.2,99.8
4,gemma-3-1B,61.7,98.3,99.6,99.9
3,Qwen3-0.6B,57.4,97.7,99.4,99.9
5,Qwen3-1.7B,66.8,96.7,99.2,99.8
1,Llama-3.2-1B,48.8,98.1,99.6,99.9
0,Llama-3.2-3B,53.5,97.3,99.2,99.8


In [ ]:
df[["model_id", "average_top2_token_probability", "recall-top-2-ef-100", "recall-top-2-ef-200", "recall-top-2-ef-400"]]

,model_id,average_top2_token_probability,recall-top-2-ef-100,recall-top-2-ef-200,recall-top-2-ef-400
2,gemma-3-270M,77.3,97.3,99.1,99.7
4,gemma-3-1B,74.2,98.0,99.5,99.9
3,Qwen3-0.6B,71.1,97.5,99.4,99.9
5,Qwen3-1.7B,79.3,96.3,99.0,99.8
1,Llama-3.2-1B,62.1,97.9,99.5,99.9
0,Llama-3.2-3B,66.8,97.1,99.2,99.8
